# Modelo de examen - Práctica

- González Viviana Elisa Soledad

In [149]:
#@title Ejecuta esta celda para cargar los datasets
"https://github.com/danteBenitez/modelo-examen-practica/raw/refs/heads/main/albums.db"
"https://github.com/danteBenitez/modelo-examen-practica/raw/refs/heads/main/tracks.json"

'https://github.com/danteBenitez/modelo-examen-practica/raw/refs/heads/main/tracks.json'

## Entrega

- Durante la actividad se pedirá que limpies el dataframe original. Al final, debes guardar el dataset limpio (con datos de las canciones y sus álbumes) en una base de datos SQLite, en una tabla `canciones`.
- El Jupyter Notebook deberá entregarse **ejecutado**, con los resultados y gráficos incluidos.
- Las justificaciones y explicaciones deberán hacerse en celdas Markdown.

# Situación problemática

El sistema de una tienda musical sufrió daños y no todos los datos se recuperaron; otros sufrieron una corrupción. Los dato están divididos en dos grupos:
- Los albumes ofrecidos en la tienda se guardan en un archivo de base de datos SQLite, en una tabla `albums`.
- Las canciones se recuperaron aparte, en un archivo `tracks.json`.

**Objetivo**: Auditar, limpiar y evaluar la calidad de los datos. Luego, elaborar un reporte con gráficos con las conclusiones que se puedan extraer de los datos limpios.

In [150]:
import pandas as pd
import sqlite3

## Actividad 1

- Realiza un diagnóstico técnico de los datos en `tracks.json`. Informa:
  - qué columnas tienen valores nulos,
  - cuál es el tipo actual de cada uno, y
  - si existen columnas que requerirán una transformación previa antes de trabajar con ellas.

In [151]:
df_canciones = pd.read_json("tracks.json")
df_canciones.head()


,AlbumId,TrackName,Milliseconds,Bytes,UnitPrice,GenreName
0,6.0,All I Really Want,284891.0,9375567,$0.99 USD,rock
1,NaN,You Oughta Know,249234.0,8196916,$0.99 USD,rock!!!
2,6.0,Perfect,188133.0,6145404,$nan,ROCK
3,6.0,Hand In My Pocket,221570.0,7224246,0.99,ROCK!!!
4,NaN,Right Through You,176117.0,5793082,$0.99 USD,rock


In [152]:
df_canciones.columns
#para traer las columnas del dataframe(propiedad)



Index(['AlbumId', 'TrackName', 'Milliseconds', 'Bytes', 'UnitPrice',
       'GenreName'],
      dtype='str')

In [153]:
df_canciones.isnull().sum()
#para saber si hay datos nulos en el dataframe(metodo)


AlbumId          9
TrackName        0
Milliseconds     0
Bytes            0
UnitPrice       17
GenreName        0
dtype: int64

In [154]:
df_canciones.dtypes
#para saber el tipo de dato de cada columna(propiedad o atributo)

AlbumId         float64
TrackName           str
Milliseconds    float64
Bytes             int64
UnitPrice           str
GenreName           str
dtype: object

In [155]:
df_canciones.info()
#para saber el tipo de datos que hay en cada columna(metodo)

<class 'pandas.DataFrame'>
Index: 631 entries, 0 to 630
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   AlbumId       622 non-null    float64
 1   TrackName     631 non-null    str    
 2   Milliseconds  631 non-null    float64
 3   Bytes         631 non-null    int64  
 4   UnitPrice     614 non-null    str    
 5   GenreName     631 non-null    str    
dtypes: float64(2), int64(1), str(3)
memory usage: 34.5 KB


- Observa los datos almacenados en el archivo SQLite. ¿Qué columna podemos usar para combinar información de ambos datasets?

In [156]:
con = sqlite3.connect("albums.db")
df_albums = pd.read_sql("select * from albums" , con)

In [157]:
df_albums.head()

,index,AlbumId,Title,ArtistId
0,0,323,Carried to Dust (Bonus Track Version),253
1,1,115,Sex Machine,91
2,2,326,Mendelssohn: A Midsummer Night's Dream,256
3,3,268,The Best of Beethoven,203
4,4,308,"Tchaikovsky: 1812 Festival Overture, Op.49, Ca...",243


## Actividad 2: Limpieza

- Los géneros musicales se grabaron de forma inconsistente (ej: "ROCK", "rock ", " Rock"). Incluso hay nombres con caracteres especiales. Explora estas inconsistencias y normaliza todos los géneros para que el inventario sea coherente.

In [158]:
df_canciones["GenreName"].unique()
#trae los generos de musica que hay en la columna GenreName(metodo)

<StringArray>
[                        'rock',                   '   rock!!!',
                         'ROCK',                   '   ROCK!!!',
                     '   rock ',                     '   ROCK ',
                        'LATIN',                    '   latin ',
                        'latin',                  '   LATIN!!!',
                    '   LATIN ',                  '   latin!!!',
                       'reggae',                 '   REGGAE!!!',
                       'REGGAE',                   '   REGGAE ',
                 '   reggae!!!',               '      ROCK !!!',
              '      latin !!!',           'ALTERNATIVE & PUNK',
     '   alternative & punk!!!',           'alternative & punk',
       '   alternative & punk ',     '   ALTERNATIVE & PUNK!!!',
         '   EASY LISTENING!!!',               'EASY LISTENING',
           '   easy listening ',               'easy listening',
         '   easy listening!!!',     '      EASY LISTENING !!!',
           

In [159]:
#eliminamos primero los espacios pq es lo mas facil jeje
df_canciones["GenreName"] = df_canciones["GenreName"].str.strip()
# strip elimina los espacios al principio y al final de cada valor de la columna GenreName(metodo)

df_canciones["GenreName"].unique()
#trae los generos de musica que hay en la columna GenreName(metodo)

<StringArray>
[                  'rock',                'rock!!!',                   'ROCK',
                'ROCK!!!',                  'LATIN',                  'latin',
               'LATIN!!!',               'latin!!!',                 'reggae',
              'REGGAE!!!',                 'REGGAE',              'reggae!!!',
               'ROCK !!!',              'latin !!!',     'ALTERNATIVE & PUNK',
  'alternative & punk!!!',     'alternative & punk',  'ALTERNATIVE & PUNK!!!',
      'EASY LISTENING!!!',         'EASY LISTENING',         'easy listening',
      'easy listening!!!',     'EASY LISTENING !!!',             'SOUNDTRACK',
             'soundtrack',          'SOUNDTRACK!!!',          'soundtrack!!!',
         'soundtrack !!!',                  'metal',               'metal!!!',
                  'METAL',               'METAL!!!',            'R&B/SOUL!!!',
               'r&b/soul',            'r&b/soul!!!',               'R&B/SOUL',
 'ALTERNATIVE & PUNK !!!',            

In [160]:
df_canciones["GenreName"] =df_canciones["GenreName"].str.replace("!", "")
df_canciones["GenreName"].unique()

<StringArray>
[               'rock',                'ROCK',               'LATIN',
               'latin',              'reggae',              'REGGAE',
               'ROCK ',              'latin ',  'ALTERNATIVE & PUNK',
  'alternative & punk',      'EASY LISTENING',      'easy listening',
     'EASY LISTENING ',          'SOUNDTRACK',          'soundtrack',
         'soundtrack ',               'metal',               'METAL',
            'R&B/SOUL',            'r&b/soul', 'ALTERNATIVE & PUNK ',
         'hip hop/rap',         'HIP HOP/RAP',               'rock ',
               'blues',               'BLUES',              'blues ',
 'alternative & punk ',            'TV SHOWS',            'tv shows',
              'METAL ',                 'pop',                 'POP',
           'classical',          'CLASSICAL ',           'CLASSICAL',
          'classical ',         'alternative']
Length: 38, dtype: str

In [161]:
df_canciones["GenreName"] = df_canciones["GenreName"].str.strip()
df_canciones["GenreName"].unique()

<StringArray>
[              'rock',               'ROCK',              'LATIN',
              'latin',             'reggae',             'REGGAE',
 'ALTERNATIVE & PUNK', 'alternative & punk',     'EASY LISTENING',
     'easy listening',         'SOUNDTRACK',         'soundtrack',
              'metal',              'METAL',           'R&B/SOUL',
           'r&b/soul',        'hip hop/rap',        'HIP HOP/RAP',
              'blues',              'BLUES',           'TV SHOWS',
           'tv shows',                'pop',                'POP',
          'classical',          'CLASSICAL',        'alternative']
Length: 27, dtype: str

In [162]:
df_canciones["GenreName"] = df_canciones["GenreName"].str.upper()
#para convertir a mayusculas los generos de musica de la columna GenreName(metodo)
df_canciones["GenreName"].unique()

<StringArray>
[              'ROCK',              'LATIN',             'REGGAE',
 'ALTERNATIVE & PUNK',     'EASY LISTENING',         'SOUNDTRACK',
              'METAL',           'R&B/SOUL',        'HIP HOP/RAP',
              'BLUES',           'TV SHOWS',                'POP',
          'CLASSICAL',        'ALTERNATIVE']
Length: 14, dtype: str

- Parece haber errores en la columna de precios. Identifica cuáles son y emplea estrategias de limpieza para solucionarlos. Listar qué problemas se encontraron.

In [163]:
df_canciones["UnitPrice"].unique()

<StringArray>
['$0.99 USD',      '$nan',      '0.99',     '$0.99',  '$nan USD',         nan,
     '$1.99', '$1.99 USD',      '1.99']
Length: 9, dtype: str

In [164]:
df_canciones["UnitPrice"] = df_canciones["UnitPrice"].str.replace("$", "")
df_canciones["UnitPrice"].unique()

<StringArray>
['0.99 USD', 'nan', '0.99', 'nan USD', nan, '1.99', '1.99 USD']
Length: 7, dtype: str

In [165]:
df_canciones["UnitPrice"] = df_canciones["UnitPrice"].str.replace("USD", "")
df_canciones["UnitPrice"].unique()

<StringArray>
['0.99 ', 'nan', '0.99', 'nan ', nan, '1.99', '1.99 ']
Length: 7, dtype: str

In [166]:
df_canciones["UnitPrice"] = df_canciones["UnitPrice"].str.strip()
df_canciones["UnitPrice"].unique()

<StringArray>
['0.99', 'nan', nan, '1.99']
Length: 4, dtype: str

In [167]:
df_canciones["UnitPrice"]= pd.to_numeric(df_canciones["UnitPrice"], errors="coerce")
df_canciones["UnitPrice"].unique()
#para convertir a numerico la columna UnitPrice(metodo), el error coerce convierte los valores que no se pueden convertir a NaN

array([0.99,  nan, 1.99])

In [168]:
df_canciones["UnitPrice"].dtype

dtype('float64')

- ¿Existen precios con valores nulos? Elige entre completar los valores con NaN, eliminar las filas o usar algún otro valor. Justifica tu elección.

In [169]:
df_canciones = df_canciones.dropna(subset=["UnitPrice"]) 
#para eliminar las filas que tienen valores nulos en la columna UnitPrice(metodo)
#subseet es para especificar la columna en la que queremos eliminar los valores nulos
df_canciones["UnitPrice"].unique()

array([0.99, 1.99])

- Se han detectado "pistas fantasma" con duraciones imposibles (menos de 10 segundos o más de 20 minutos). Elimínalas para que no afecten las estadísticas de la tienda.

In [170]:
indices_mayores = df_canciones[df_canciones["Milliseconds"] > 1200000].index
#1200000 milisegundos es igual a 20 minutos, entonces estamos buscando las canciones que duran mas de 20 minutos
#almacenamos los indices de las filas que tienen un valor mayor a 1200000 en la columna Milliseconds(metodo)

In [171]:
indices_menores = df_canciones[df_canciones["Milliseconds"] < 10000].index
#10000 milisegundos es igual a 10 segundos, entonces estamos buscando las canciones que duran menos de 10 segundos
#almacenamos los indices de las filas que tienen un valor menor a 10000 en la columna Milliseconds(metodo)

In [172]:
df_canciones = df_canciones.drop(index=indices_mayores)

In [173]:
df_canciones = df_canciones.drop(index= indices_menores)

In [174]:
df_canciones[df_canciones["Milliseconds"] > 1200000].index

Index([], dtype='int64')

In [175]:
df_canciones[df_canciones["Milliseconds"] < 10000].index   

Index([], dtype='int64')

## Actividad 3: Reconstrucción

- **Reconstruye el inventario**. Combina la información de los albumes con sus canciones, utilizando la columna identificada en la Actividad 1.

**Pista**: Deberás usar la función `pd.merge()` de Pandas. De la siguiente manera:

```py
combined = pd.merge(df1, df2, on='<columna>')
```

- ¿Qué cantidad de registros esperas que tenga el dataframe obtenido? Verifica si se cumple o no tu expectativa.

In [176]:
combined = pd.merge(df_canciones, df_albums, on="AlbumId")
combined.columns

Index(['AlbumId', 'TrackName', 'Milliseconds', 'Bytes', 'UnitPrice',
       'GenreName', 'index', 'Title', 'ArtistId'],
      dtype='str')

- ¿Todos las canciones pertenecen a un album? Si es posible, calcula cuánto dinero representan esas canciones perdidas.

In [177]:
df_canciones.shape
#para saber la cantidad de filas y columnas que tiene el dataframe(propiedad)

(564, 6)

In [178]:
df_perdido = df_canciones[df_canciones["AlbumId"].isnull()]

In [179]:
#para sumar de la variable dfperdido la cantidad de unit price que hay en la columna UnitPrice(metodo)
df_perdido["UnitPrice"].sum()

np.float64(7.92)

In [180]:
combined.shape

(556, 9)

In [ ]:
combined.groupby('Title')['TrackName'].count()
#groupby agrupa el dataframe por la columna TITLE y luego se basa en la columna trackname paara contar la cantidad de canciones que hay por cada album(metodo)

Title
A Copland Celebration, Vol. I                                                                       1
A Soprano Inspired                                                                                  1
A-Sides                                                                                            15
Acústico                                                                                           19
Acústico MTV [Live]                                                                                16
Album Of The Year                                                                                  12
Are You Experienced?                                                                               16
As Canções de Eu Tu Eles                                                                           13
Audioslave                                                                                         14
Bach: The Brandenburg Concertos                                             

- Agrupa los datos por Title. Calcula:
    - El número de canciones por álbum.
    - El precio total del álbum.
    - El tamaño total en Megabytes.

 - ¿Cuál es el álbum más caro de la tienda según tus cálculos?

In [ ]:
combined.groupby("Title")["UnitPrice"].sum()
#groupby agrupa el dataframe por la columna TITLE y luego se basa en la columna UnitPrice para sumar el precio total de las canciones por cada album(metodo)

Title
A Copland Celebration, Vol. I                                                                       0.99
A Soprano Inspired                                                                                  0.99
A-Sides                                                                                            14.85
Acústico                                                                                           18.81
Acústico MTV [Live]                                                                                15.84
Album Of The Year                                                                                  11.88
Are You Experienced?                                                                               15.84
As Canções de Eu Tu Eles                                                                           12.87
Audioslave                                                                                         13.86
Bach: The Brandenburg Concertos                  

In [185]:
#1byte
#1kb = 1024 bytes
#1mb = 1024 kb

(combined.groupby('Title')['Bytes'].sum()/ (1024*1024)).round(2)
#groupby agrupa el dataframe por la columna TITLE y luego se basa en la columna Bytes

Title
A Copland Celebration, Vol. I                                                                        3.06
A Soprano Inspired                                                                                   5.35
A-Sides                                                                                            130.81
Acústico                                                                                           117.10
Acústico MTV [Live]                                                                                117.99
Album Of The Year                                                                                   80.76
Are You Experienced?                                                                               107.22
As Canções de Eu Tu Eles                                                                            85.74
Audioslave                                                                                          89.94
Bach: The Brandenburg Concertos         

In [186]:
#valor mas caro
combined.groupby("Title")["UnitPrice"].sum().idxmax()
#el idxmax devuelve el indice del valor maximo de la columna UnitPrice, que en este caso es el titulo del album mas caro

'Greatest Hits'

- Calcula el valor total del inventario (suma de UnitPrice) y la duración promedio de las canciones para cada género.

In [188]:
combined["UnitPrice"].sum()

np.float64(550.44)

In [189]:
(combined.groupby("GenreName")["Milliseconds"].mean()/100).round(2)

GenreName
ALTERNATIVE           2153.86
ALTERNATIVE & PUNK    2093.29
BLUES                 3255.51
CLASSICAL             2872.94
EASY LISTENING        1893.84
HIP HOP/RAP           1924.49
LATIN                 2290.16
METAL                 3155.06
POP                   2420.12
R&B/SOUL              2432.56
REGGAE                2469.97
ROCK                  2837.14
SOUNDTRACK            2333.23
Name: Milliseconds, dtype: float64

## Actividad 4: Gráficos

Responde a las siguientes actividades con un gráfico.
Asegúrate de:
- Elegir un título de gráfico adecuado.
- Elegir el tipo de gráfico correcto para lo que se quiere representar.
- Añadir etiquetas de eje y leyendas, de ser necesarias.

- ¿Qué géneros tenemos más en stock?

- ¿Cómo se distribuyen los precios de las canciones según su género?

- ¿Hay relación entre la duración de la canción y el género?